# arch_est

> 对应代码：`EconometricsCode/code_in_notes/Chap.28/arch_est.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.28`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.28")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们先做环境设置并读入股价数据，保留股票代码为 600009 的样本，按日期排序后生成行号 t 作为时间索引。由于非交易日会造成日历日期不连续，用行号 t 做时间标识（tsset）更稳妥。

In [ ]:
%%stata
clear all
set more off
use ../datasets/stock_price
keep if stkcd==600009
sort date
gen t=_n
tsset stkcd t

我们计算对数收益率 rr = Δlog(P)，即相邻两期收盘价对数之差。这是金融计量中最常用的收益率度量，具有近似可加性和平稳性。

In [ ]:
%%stata
gen rr=log(clsprc)-log(L.clsprc)
label variable rr "对数收益率"

画出对数收益率的时序图，直观观察波动聚集（volatility clustering）现象。

In [ ]:
%%stata
line rr date

我们估计一个 ARCH(5) 模型：条件方差 σ²_t = ω + Σ αᵢ ε²_{t-i}（i=1..5），即用过去 5 期残差平方的线性组合来刻画条件异方差。Stata 的 arch 命令默认均值方程为常数，arch(1/5) 指定滞后 1 到 5 阶的 ARCH 项。估计后用 predict 的 variance 选项得到条件方差序列，开根号即条件标准差（波动率）。

In [ ]:
%%stata
// ARCH(5)
arch rr, arch(1/5)
predict sigma2_arch, variance
gen sigma_arch=sqrt(sigma2_arch)
label variable sigma_arch "ARCH(5)"

GARCH(1,1) 在 ARCH(1) 基础上加入一期滞后的条件方差项：σ²_t = ω + α ε²_{t-1} + β σ²_{t-1}。β 项使波动具有持续性，通常用很少参数就能捕捉高阶 ARCH 效应，是应用最广泛的波动率模型。

In [ ]:
%%stata
// GARCH(1,1)
arch rr, arch(1) garch(1)
predict sigma2_garch, variance
gen sigma_garch=sqrt(sigma2_garch)
label variable sigma_garch "GARCH(1,1)"

EGARCH(1,1) 对 log σ²_t 建模，允许正负冲击（利好消息 vs 利空消息）对波动产生非对称影响（杠杆效应），且无需对参数施加非负约束。earch(1) 是标准化的 ARCH 项，egarch(1) 是滞后对数方差项。

In [ ]:
%%stata
// EGARCH(1,1)
arch rr, earch(1) egarch(1)
predict sigma2_egarch, variance
gen sigma_egarch=sqrt(sigma2_egarch)
label variable sigma_egarch "EGARCH(1,1)"

GJR 门限 ARCH（threshold ARCH）在 GARCH(1,1) 基础上加入 tarch(1) 项：当滞后残差为负时额外增加一个系数，从而捕捉坏消息比好消息引起更大波动的非对称性。

In [ ]:
%%stata
// GJR threshold arch
arch rr, arch(1) garch(1) tarch(1)
predict sigma2_tgarch, variance
gen sigma_tgarch=sqrt(sigma2_tgarch)
label variable sigma_tgarch "GJR"

重新按日历日期设定时间索引，把四个模型估计出的条件标准差画在同一张图上比较，legend 选项调整图例位置以免遮挡曲线。

In [ ]:
%%stata
// 画图
tsset date
tsline sigma_*, legend(ring(0) position(11) bmargin(large))

GARCH-M（GARCH-in-mean）把条件标准差（或方差）直接放入均值方程，刻画「风险越大、要求收益越高」的风险溢价机制。archm 选项即让 σ_t 进入收益率方程。

In [ ]:
%%stata
// GARCH-M
arch rr, arch(1) garch(1) archm